# Omni-VLA: train BC on Kaggle GPU
Upload the source archive and `drawer_dagger_v1.h5` as private Kaggle datasets, import this notebook, attach the datasets, and enable GPU + Internet. Default: BC only. This is the privileged state baseline, not end-to-end VLA.


In [ ]:
from pathlib import Path
import zipfile, sys, subprocess, shutil
import torch

POLICIES = ("bc",)  # Add "cfm" later when ready to compare policies.
INPUT = Path("/kaggle/input")
REPO = Path("/kaggle/working/omni_vla")
print(torch.__version__, "CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable GPU in Notebook Settings > Accelerator"
print("GPU:", torch.cuda.get_device_name(0))

# Kaggle may expose the ZIP itself or unpack its contents during dataset upload.
archives = list(INPUT.rglob("kaggle_source.zip"))
if archives:
    assert len(archives) == 1, "Attach exactly one source archive"
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall("/kaggle/working")
else:
    candidates = [p.parent for p in INPUT.rglob("pyproject.toml")
                  if (p.parent / "scripts/train_kaggle.py").is_file()]
    assert len(candidates) == 1, "Attach the Omni-VLA source dataset (ZIP or unpacked files)"
    shutil.copytree(candidates[0], REPO, dirs_exist_ok=True)
assert (REPO / "scripts/train_kaggle.py").is_file(), "Source layout is incomplete"
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO)+"[train,deploy]"], check=True)


In [ ]:
datasets = list(INPUT.rglob("drawer_dagger_v1.h5"))
assert len(datasets) == 1, "Attach exactly one drawer_dagger_v1.h5 dataset"
DATA = datasets[0]
print("Dataset:", DATA)
subprocess.run([sys.executable, "-m", "scripts.validate_dataset", str(DATA)], cwd=REPO, check=True)


In [ ]:
for policy in POLICIES:
    subprocess.run([sys.executable, "-m", "scripts.train_kaggle", "--dataset", str(DATA), "--policy", policy, "--config", str(REPO/("configs/train_bc_kaggle.yaml" if policy == "bc" else "configs/train_cfm_kaggle.yaml")), "--output", f"/kaggle/working/{policy}"], cwd=REPO, check=True)


In [ ]:
for policy in POLICIES:
    subprocess.run([sys.executable, "-m", "scripts.export_policy", "--checkpoint", f"/kaggle/working/{policy}/best.pt", "--output", f"/kaggle/working/{policy}/policy.onnx"], cwd=REPO, check=True)


In [ ]:
for policy in POLICIES:
    archive = shutil.make_archive(f"/kaggle/working/{policy}_outputs", "zip",
                                  root_dir="/kaggle/working", base_dir=policy)
    print("Download:", archive)
    print("Checkpoint:", Path(f"/kaggle/working/{policy}/best.pt"))


Save Version with outputs, then download `bc_outputs.zip` from notebook outputs. It contains the `bc/` folder with checkpoints, metrics, ONNX and JSON metadata. Extract it on Dell and place the folder at `weights/bc_kaggle/`. GPU training has not been executed from the local development session.
